Cell 1 — Imports & Config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType, ArrayType
from datetime import datetime
import json

# --- Config ---
STORAGE_ACCOUNT = "stflightliabilitydev01"
RAW_CONTAINER = "raw"
RAW_BASE_PATH = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/airlabs/delays"
CHECKPOINT_PATH = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/bronze_delays"
QUARANTINE_BASE_PATH = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_quarantine/bronze_delays"

CATALOG = "cat_flight_liability"
SCHEMA = "db_bronze"
TABLE = "arrival_delays"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

Cell 2 — Logger

In [0]:
def log(level, message, **kwargs):
    print(json.dumps({"timestamp": datetime.utcnow().isoformat(), "level": level, "message": message, **kwargs}))

log("INFO", "Bronze run started")

{"timestamp": "2026-09-02T02:40:36.017295", "level": "INFO", "message": "Bronze run started"}


/home/spark-c02166a3-9946-45f0-989f-2f/.ipykernel/89/command-5756936834370503-2209051223:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp": datetime.utcnow().isoformat(), "level": level, "message": message, **kwargs}))


Cell 3 — Explicit schema definition

In [0]:
# iata stands for International Air Transport Association
# icao stands for International Civil Aviation Organization

schedule_record_schema = StructType([
    StructField("airline_iata", StringType(), True),
    StructField("airline_icao", StringType(), True),
    StructField("flight_iata", StringType(), True),
    StructField("flight_icao", StringType(), True),
    StructField("flight_number", StringType(), True),
    StructField("cs_airline_iata", StringType(), True),
    StructField("cs_flight_number", StringType(), True),
    StructField("cs_flight_iata", StringType(), True),

    StructField("dep_iata", StringType(), True),
    StructField("dep_icao", StringType(), True),
    StructField("dep_terminal", StringType(), True),
    StructField("dep_gate", StringType(), True),
    StructField("dep_time", StringType(), True),
    StructField("dep_time_ts", LongType(), True),
    StructField("dep_time_utc", StringType(), True),
    StructField("dep_estimated", StringType(), True),
    StructField("dep_estimated_ts", LongType(), True),
    StructField("dep_estimated_utc", StringType(), True),
    StructField("dep_actual", StringType(), True),
    StructField("dep_actual_ts", LongType(), True),
    StructField("dep_actual_utc", StringType(), True),

    StructField("arr_iata", StringType(), True),
    StructField("arr_icao", StringType(), True),
    StructField("arr_terminal", StringType(), True),
    StructField("arr_gate", StringType(), True),
    StructField("arr_baggage", StringType(), True),
    StructField("arr_time", StringType(), True),
    StructField("arr_time_ts", LongType(), True),
    StructField("arr_time_utc", StringType(), True),
    StructField("arr_estimated", StringType(), True),
    StructField("arr_estimated_ts", LongType(), True),
    StructField("arr_estimated_utc", StringType(), True),
    StructField("arr_actual", StringType(), True),
    StructField("arr_actual_ts", LongType(), True),
    StructField("arr_actual_utc", StringType(), True),

    StructField("duration", IntegerType(), True),
    StructField("delayed", IntegerType(), True),        
    StructField("dep_delayed", IntegerType(), True),
    StructField("arr_delayed", IntegerType(), True),
    StructField("status", StringType(), True),
])

# here only definining schema field for has_more because that is the only one we need

request_meta_schema = StructType([
    StructField("has_more", BooleanType(), True),
])


# reponse is got as an array of objects so in response_schema ArrayType(schedule_record_schema)
response_schema = StructType([
    StructField("request", request_meta_schema, True),
    StructField("response", ArrayType(schedule_record_schema), True)
])

Cell 4 — Read raw JSON with explicit schema enforced

In [0]:
# corrupt record is a column that is build automatically when the file is read on pemrmissive mode
# permissive mode is when if the data that is read from the file doesn't match the schema we defined
# permissive mode takes that record into separate column which is corrupt_record
# Above only require when using batch processing without autoload when using autoload that handles it
raw_df = (
    spark.readStream
    # this says I am watching via autoloaders cloud watching system and has nothing to do with our file format the format doesnt change
    # its just naming how we are detecting or reading files
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")
    .option("cloudFiles.schemaLocation",CHECKPOINT_PATH)
    .option("cloudFiles.rescuedDataColumn","rescued_data")
    .schema(response_schema)
    .load(f"{RAW_BASE_PATH}/*/*.json")
    .withColumn("_sourcefile" , F.col("_metadata.file_path"))
)
log("INFO", "Raw files read", source_path = RAW_BASE_PATH , checkpoint_path = CHECKPOINT_PATH)

/home/spark-c02166a3-9946-45f0-989f-2f/.ipykernel/89/command-5756936834370503-2209051223:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp": datetime.utcnow().isoformat(), "level": level, "message": message, **kwargs}))


{"timestamp": "2026-09-02T02:40:36.534467", "level": "INFO", "message": "Raw files read", "source_path": "abfss://raw@stflightliabilitydev01.dfs.core.windows.net/airlabs/delays", "checkpoint_path": "abfss://raw@stflightliabilitydev01.dfs.core.windows.net/_checkpoints/bronze_delays"}


Cell 5 : Process Batch


In [0]:
# def process_batch(batch_df , batch_id):
#     print(">>> process_batch WAS CALLED <<<")
#     batch_row_count = batch_df.count()
#     if batch_row_count == 0 :
#         log("INFO","No data found to process" , batch_id = batch_id)
#         return
#     log("INFO","Processing Microbatch", batch_id=batch_id , top_level_docs = batch_row_count)

#     rescued_records = batch_df.filter(F.col("rescued_data").isNotNull())
#     rescued_records_count = rescued_records.count()
#     if rescued_records_count > 0:
#         log("WARNING" , "rescued records found qurantining" , batch_id = batch_id , rescued_record_count = rescued_records_count)
#         quarantine_path = f"{QUARANTINE_BASE_PATH}/{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_batch_{batch_id}"
#         rescued_records.write.mode("overwrite").json(quarantine_path)
#     batch_df = batch_df.filter(F.col("rescued_data").isNull())

#     flights_df = batch_df.select(F.explode("response").alias("flight"), "_sourcefile")

#     TS_FORMAT = "yyyy-MM-dd HH:mm"
#     bronze_batch_df = flights_df.select(
#         F.col("flight.airline_iata").alias("airline_iata"),
#         F.col("flight.airline_icao").alias("airline_icao"),
#         F.col("flight.flight_iata").alias("flight_iata"),
#         F.col("flight.flight_icao").alias("flight_icao"),
#         F.col("flight.flight_number").alias("flight_number"),

#         F.col("flight.dep_iata").alias("dep_iata"),
#         F.col("flight.dep_icao").alias("dep_icao"),
#         F.col("flight.dep_terminal").alias("dep_terminal"),
#         F.col("flight.dep_gate").alias("dep_gate"),
#         F.to_timestamp("flight.dep_time", TS_FORMAT).alias("dep_scheduled"),
#         F.to_timestamp("flight.dep_estimated", TS_FORMAT).alias("dep_estimated"),
#         F.to_timestamp("flight.dep_actual", TS_FORMAT).alias("dep_actual"),
#         F.col("flight.dep_delayed").alias("dep_delay_minutes"),

#         F.col("flight.arr_iata").alias("arr_iata"),
#         F.col("flight.arr_icao").alias("arr_icao"),
#         F.col("flight.arr_terminal").alias("arr_terminal"),
#         F.col("flight.arr_gate").alias("arr_gate"),
#         F.col("flight.arr_baggage").alias("arr_baggage"),
#         F.to_timestamp("flight.arr_time", TS_FORMAT).alias("arr_scheduled"),
#         F.to_timestamp("flight.arr_estimated", TS_FORMAT).alias("arr_estimated"),
#         F.to_timestamp("flight.arr_actual", TS_FORMAT).alias("arr_actual"),
#         F.col("flight.arr_delayed").alias("arr_delay_minutes"),

#         F.col("flight.duration").alias("duration_minutes"),
#         F.col("flight.status").alias("flight_status"),

#         F.col("_sourcefile"),
#         F.current_timestamp().alias("_ingested_at"),
#         F.regexp_extract(F.col("_sourcefile"), r"ingest_date=([0-9\-]+)", 1).alias("_pull_ingest_date"),
#     )

#     record_count = bronze_batch_df.count()
#     log("INFO", "Parsed flight records", batch_id=batch_id, count=record_count)

#     if record_count > 0:
#         null_flight_iata = bronze_batch_df.filter(F.col("flight_iata").isNull()).count()
#         if null_flight_iata / record_count > 0.05:
#             log("WARNING", "High null rate on flight_iata", batch_id=batch_id,
#                 null_flight_iata=null_flight_iata, total=record_count)
#     (
#         bronze_batch_df.write
#         .format("delta")
#         .mode("append")
#         .option("mergeSchema", "true")
#         .saveAsTable(BRONZE_TABLE)
#     )
#     log("INFO", "Batch write complete", batch_id=batch_id, table=BRONZE_TABLE, rows_written=record_count)

DEBUG_OUTPUT_PATH = (
    f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "_debug/bronze_delays_rescued"
)


def process_batch(batch_df, batch_id):

    rescued_df = (
        batch_df
        .filter(F.col("rescued_data").isNotNull())
        .select(
            "rescued_data",
            "_sourcefile"
        )
    )

    (
        rescued_df
        .write
        .mode("overwrite")
        .json(DEBUG_OUTPUT_PATH)
    )

Cell 6 — Inititating the Stream

In [0]:
# query = (
#     raw_df.writeStream
#     .foreachBatch(process_batch)
#     .option("checkpointLocation", CHECKPOINT_PATH)
#     # below code is what makes the stream batch meaning 
#     .trigger(availableNow=True)
#     .start()
# )

# query.awaitTermination()
# log("INFO", "Bronze run completed — all available files processed")

DEBUG_CHECKPOINT_PATH = (
    f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "_checkpoints/bronze_delays_debug_4"
)

query = (
    raw_df.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", DEBUG_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

log("INFO", "Debug rescued-data run completed")

{"timestamp": "2026-09-02T04:24:31.343478", "level": "INFO", "message": "Debug rescued-data run completed"}


/home/spark-c02166a3-9946-45f0-989f-2f/.ipykernel/89/command-5756936834370503-2209051223:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(json.dumps({"timestamp": datetime.utcnow().isoformat(), "level": level, "message": message, **kwargs}))


In [0]:
# display(
#     spark.read
#     .json(DEBUG_OUTPUT_PATH)
# )

test_df = (
    spark.read
    .format("json")
    .option("multiline", "true")
    .schema(response_schema)
    .load(f"{RAW_BASE_PATH}/*/*.json")
)

display(
    test_df
    .select(F.explode("response").alias("flight"))
    .select("flight.*")
    .limit(5)
)



airline_iata,airline_icao,flight_iata,flight_icao,flight_number,cs_airline_iata,cs_flight_number,cs_flight_iata,dep_iata,dep_icao,dep_terminal,dep_gate,dep_time,dep_time_ts,dep_time_utc,dep_estimated,dep_estimated_ts,dep_estimated_utc,dep_actual,dep_actual_ts,dep_actual_utc,arr_iata,arr_icao,arr_terminal,arr_gate,arr_baggage,arr_time,arr_time_ts,arr_time_utc,arr_estimated,arr_estimated_ts,arr_estimated_utc,arr_actual,arr_actual_ts,arr_actual_utc,duration,delayed,dep_delayed,arr_delayed,status
DL,DAL,DL233,DAL233,233,null,null,null,NAP,LIRN,M,B15,2026-08-16 10:15,1786868100,2026-08-16 08:15,2026-08-16 10:11,1786867860,2026-08-16 08:11,2026-08-16 10:11,1786867860,2026-08-16 08:11,JFK,KJFK,4,B20,null,2026-08-16 14:34,1786905240,2026-08-16 18:34,2026-08-16 11:00,1786870800,2026-08-16 09:00,2026-08-16 11:00,1786870800,2026-08-16 09:00,619,115,160,115,active
AF,AFR,AF9109,AFR9109,9109,DL,233,DL233,NAP,LIRN,M,B15,2026-08-16 10:15,1786868100,2026-08-16 08:15,2026-08-16 10:11,1786867860,2026-08-16 08:11,2026-08-16 10:11,1786867860,2026-08-16 08:11,JFK,KJFK,4,B20,null,2026-08-16 14:34,1786905240,2026-08-16 18:34,2026-08-16 11:00,1786870800,2026-08-16 09:00,2026-08-16 11:00,1786870800,2026-08-16 09:00,619,115,160,115,active
KL,KLM,KL6172,KLM6172,6172,DL,233,DL233,NAP,LIRN,M,B15,2026-08-16 10:15,1786868100,2026-08-16 08:15,2026-08-16 10:11,1786867860,2026-08-16 08:11,2026-08-16 10:11,1786867860,2026-08-16 08:11,JFK,KJFK,4,B20,null,2026-08-16 14:34,1786905240,2026-08-16 18:34,2026-08-16 11:00,1786870800,2026-08-16 09:00,2026-08-16 11:00,1786870800,2026-08-16 09:00,619,115,160,115,active
SN,BEL,SN9633,BEL9633,9633,AC,871,AC871,CDG,LFPG,2C,A39,2026-08-16 14:10,1786882200,2026-08-16 12:10,2026-08-16 14:18,1786882680,2026-08-16 12:18,2026-08-16 14:18,1786882680,2026-08-16 12:18,YUL,CYUL,null,null,null,2026-08-16 15:45,1786909500,2026-08-16 19:45,2026-08-16 15:00,1786885200,2026-08-16 13:00,2026-08-16 15:00,1786885200,2026-08-16 13:00,455,108,8,108,active
OU,CTN,OU5930,CTN5930,5930,AC,871,AC871,CDG,LFPG,2C,A39,2026-08-16 14:10,1786882200,2026-08-16 12:10,2026-08-16 14:18,1786882680,2026-08-16 12:18,2026-08-16 14:18,1786882680,2026-08-16 12:18,YUL,CYUL,null,null,null,2026-08-16 15:45,1786909500,2026-08-16 19:45,2026-08-16 15:00,1786885200,2026-08-16 13:00,2026-08-16 15:00,1786885200,2026-08-16 13:00,455,108,8,108,active
